## 1. Import Python Libraries

An essential first step is to import the various Python libraries needed for the project.  For example:

```python
# python imports
from crim_intervals import * 
from crim_intervals import importScore 
from crim_intervals import main_objs
from community import community_louvain
```




In [14]:
# python imports
from crim_intervals import * 
from crim_intervals import importScore 
from crim_intervals import main_objs
from community import community_louvain
from copy import deepcopy
from IPython.display import SVG
from collections import Counter
import altair as alt
import glob as glob
import crim_intervals
import crim_intervals.visualizations as viz
from crim_code import *
import numpy as np
import math

import plotly.express as px
import plotly.io as pio


import os
import pandas as pd
import re
import networkx as nx
import requests
from itertools import combinations


import base64
from IPython.display import Image, display
MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    # print("created folder : ", MYDIR)

# else:
#     print(MYDIR, "folder already exists.")
    
MUSDIR = ("MEI")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    # print("created folder : ", MUSDIR)

# else:
#     print(MUSDIR, "folder already exists.")

# code for Quarto and the charting tools
alt.renderers.enable('default')
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

import warnings
warnings.filterwarnings('ignore')
    

## 1b.  Import Functions

We also have various functions used in the steps below.  **Do not edit these!** Just run the cell below!

# 2. Import a Score

Here we find a piece to explore.  

Load it remotely:

```
filename = 'https://crimproject.org/mei/CRIM_Model_0019.mei'

piece = importScore(file_name)
```


In [15]:
filename = 'https://crimproject.org/mei/CRIM_Model_0019.mei'

piece = importScore(filename)
piece.metadata

{'title': 'Veni sponsa Christi',
 'composer': 'Giovanni Pierluigi da Palestrina',
 'date': None}

# 3 Ngrams:  Melodic

### 3a. Here are all the mel ngs.

In [16]:

mel = piece.melodic(kind='d', end=False)
mel = piece.numberParts(df=mel)
mel_ng = piece.ngrams(df=mel, n=3)
# clean up the tuples as strings
mel_ng_clean = mel_ng.map(convert_tuple)
mel_ng_clean

,1,2,3,4
0.0,-3_3_2,,,
8.0,3_2_-2,-3_3_2,,
12.0,2_-2_-3,,,
16.0,-2_-3_2,3_2_-2,,
20.0,-3_2_2,2_-2_-3,,
...,...,...,...,...
514.0,-2_-2_-2,,3_-2_-2,
516.0,,-2_-2_1,-2_-2_-2,
517.0,-2_-2_1,,,
518.0,-2_1_-2,,,


### 3b. How Many of Each ngram?

In [17]:
mel_ng_melt = mel_ng_clean.melt()
mel_ng_melt_clean = mel_ng_melt[mel_ng_melt['value'] != '']
mel_ng_counts = pd.DataFrame(mel_ng_melt_clean['value'].value_counts()).reset_index()

# the ones that are not unique (and thus repeat)
mel_ng_counts[mel_ng_counts['count'] > 1].head(10)

,value,count
0,2_2_2,63
1,-2_-2_-2,49
2,2_2_-2,42
3,-2_-2_2,40
4,2_-2_-2,31
5,-3_2_2,27
6,-2_2_2,24
7,3_-2_-2,16
8,-2_-3_2,13
9,2_-2_-3,12


## 3c. Where are These Ngrams?

Can we begin to think about where the **fugas** are based on the **offset differences** between the ngrams?  Are the fugas likely to share the same time intervals between entries?

Based on what you see here, what do you **expect to find in the score**?

In [18]:

mel = piece.melodic(kind='d', end=False)
mel = piece.numberParts(df=mel)
mel_ng = piece.ngrams(df=mel, n=3)
# clean up the tuples as strings
mel_ng_clean = mel_ng.map(convert_tuple)


# stack all the ngs
mel_ng_stacked = mel_ng_clean.stack()

# find just one of the ngs
selected_ng = mel_ng_stacked[mel_ng_stacked == '3_2_2']

# use detailIndex to report offset, measure, and beat information
selected_ng_locations = piece.detailIndex(df=selected_ng.unstack().fillna(''), offset=True)
selected_ng_locations

,,,3
Measure,Beat,Offset,
46.0,1.0,360.0,3_2_2


## 3d. Ngrams in Time

Here we show the ngrams as the appear in time, from the beginning (at the left) to the end (at the right) of the motet. 

- Each **ngram** is associated with a distinct color, and the relative placement of the ngrams among the voices can be used to imagine the kinds of contrapuntal patterns heard at each moment in the piece.

- The heatmap charts are dynamic:  you can click on a given color to isolate that ngram to see where it appears in the piece.

- Note:  some colors repeat or are hard to distinguish from each other.  But the 'key' at the top and right make it clear which color corresponds to which ngram.

As a reminder, here are the ngram settings used:

```python
entries_only = True
combine_unisons_choice = False
kind_choice ='d'
directed = True
compound = False
length_choice = 4
include_count = False
```

Then to run the maps for any set of files:


```python
for file in file_list:
    piece = importScore(file)

    chart = ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count)
    print(piece.metadata)
    display(chart)
```

In [19]:

# the ngram conditions
entries_only = True
combine_unisons_choice = False
kind_choice ='d'
directed = True
compound = False
length_choice = 3
include_count = True

# calling the function on your file list:

# for filename in my_pieces:
filename = 'https://crimproject.org/mei/CRIM_Model_0019.mei'
piece = importScore(filename)

chart = ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count, entries_only)


print(piece.metadata)
display(chart)
        
    

{'title': 'Veni sponsa Christi', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': None}


alt.VConcatChart(...)

# 4. Presentation Types with Linked Examples

Here we use another tool to identify the Presentation Types formed by the ngrams, according to Peter Schubert's useful taxonomy of Periodic Entries, Imitative Duos, and Fugas.

We also return these with useful metadata and the possibility to show them as engraved verovio scores with highlights.



In [20]:
# here we load

baseUrl = 'https://crimproject.org/mei/'

# now specify YOUR piece:
my_piece = 'CRIM_Model_0019.mei'


# and combine them
piece_url = baseUrl + my_piece

# load the piece from the web
piece = importScore(piece_url)

# find p types
pt = piece.presentationTypes(kind='d',
                             limit_to_entries = True,
                        head_flex=0,
                        body_flex = 0,
                        include_hidden_types = False,
                        combine_unisons = False,
                        melodic_ngram_length = 3)

# reorganize columns
cols = ['Composer', 'Title', 'Presentation_Type', 'Soggetti', 'Voices', 
'Number_Entries', 'Measures_Beats', 'Melodic_Entry_Intervals', 'Time_Entry_Intervals',	
'Flexed_Entries', 'Parallel_Entries','Parallel_Voice', 'Count_Non_Overlaps', 'Offsets', 'Progress']

pt = pt[cols]

# and return the results with a link to the example
piece.linkExamples(df = pt, piece_url=piece_url)


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F1%2C11%2C12%2C13%2C14%2C15%2C16%2C17%2C18%2C19%2C2%2C3%2C4%2C6%2C7%2C8%2C9%2F1%2C2%2C2%2C2%2C1%2C1%2C1%2C4%2C4%2C4%2C1%2B2%2C1%2B2%2C2%2C3%2C3%2B4%2C3%2B4%2C4%2F%40all%2C%403.0-end%2C%40all%2C%401-3.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2C%40all%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2Fhighlight&measure_range=1-19&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus, Altus, Cantus, Bassus]",7,"[1.0/1.0, 2.0/1.0, 6.0/1.0, 7.0/1.0, 11.0/3.0, 14.0/1.0, 17.0/1.0]","[P-5, P-4, P-5, P8, P5, P-8]","[8.0, 32.0, 8.0, 36.0, 20.0, 24.0]",False,False,None,0,"[0.0, 8.0, 40.0, 48.0, 84.0, 104.0, 128.0]",0.000000
1,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F19%2C20%2C21%2C22%2C23%2C25%2C26%2C27%2C28%2C29%2C30%2C32%2C33%2C35%2C36%2F3%2C3%2C1%2C1%2B2%2C2%2C3%2C3%2C4%2C4%2C2%2C2%2C1%2C1%2C4%2C4%2F%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2B%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%404.0-end%2C%401-4.0%2C%402.0-end%2C%401-2.0%2C%402.0-end%2C%401-2.0%2Fhighlight&measure_range=19-36&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 2, 2)]","[Tenor, Cantus, Altus, Tenor, Bassus, Altus, Cantus, Bassus]",8,"[19.0/4.0, 21.0/4.0, 22.0/4.0, 25.0/4.0, 27.0/4.0, 29.0/4.0, 32.0/2.0, 35.0/2.0]","[P8, P-5, P-4, P-5, P8, P8, P-11]","[16.0, 8.0, 24.0, 16.0, 16.0, 20.0, 24.0]",False,False,None,0,"[150.0, 166.0, 174.0, 198.0, 214.0, 230.0, 250.0, 274.0]",0.284091
2,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,FUGA,"[(1, 1, -3)]","[Altus, Tenor, Cantus, Bassus, Altus, Cantus, Bassus]",7,"[38.0/2.0, 38.0/4.0, 40.0/2.0, 42.0/2.0, 44.0/1.0, 46.0/2.0, 48.0/4.0]","[P-5, P8, P-12, P8, P8, P-11]","[4.0, 12.0, 16.0, 14.0, 18.0, 20.0]",False,False,None,0,"[298.0, 302.0, 314.0, 330.0, 344.0, 362.0, 382.0]",0.564394
3,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F50%2C51%2C52%2C53%2C54%2C55%2C56%2C57%2C59%2C60%2C61%2C62%2C63%2C64%2C65%2F2%2C1%2B2%2C1%2C4%2C4%2C3%2C2%2B3%2C2%2C4%2C4%2C1%2C1%2B2%2C2%2C3%2C3%2F%402.0-end%2C%404.0-end%2B%401-2.0%2C%401-4.0%2C%402.0-end%2C%401-2.0%2C%402.0-end%2C%404.0-end%2B%401-2.0%2C%401-4.0%2C%40all%2C%401-2.0%2C%404.0-end%2C%401-4.0%2B%402.0-end%2C%401-2.0%2C%402.0-end%2C%401-2.0%2Fhighlight&measure_range=50-65&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-2, 2, 1)]","[Altus, Cantus, Bassus, Tenor, Altus, Bassus, Cantus, Altus, Tenor]",9,"[50.0/2.0, 51.0/4.0, 53.0/2.0, 55.0/2.0, 56.0/4.0, 59.0/1.0, 61.0/4.0, 62.0/2.0, 64.0/2.0]","[P5, P-8, P4, P5, P-8, P8, P-4, P-5]","[12.0, 12.0, 16.0, 12.0, 18.0, 22.0, 4.0, 16.0]",False,False,None,0,"[394.0, 406.0, 418.0, 434.0, 446.0, 464.0, 486.0, 490.0, 506.0]",0.746212


In [21]:
cads = piece.cadences()
cads['Composer'] = piece.metadata['composer']
cads['Title'] = piece.metadata['title']

cads[['Composer', 'Title', 'CadType', 'Tone', 'CVFs', 'Measure', 'Beat', 'Progress']]




,Composer,Title,CadType,Tone,CVFs,Measure,Beat,Progress
Last,,,,,,,,
48.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Clausula Vera,G,TCu,7.0,1.0,0.090909
84.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,G,tC,11.0,3.0,0.159091
136.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Quince,G,QC,18.0,1.0,0.257576
152.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Authentic,G,TCB,20.0,1.0,0.287879
224.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Abandoned Clausula Vera,C,zC,29.0,1.0,0.424242
252.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,C,tC,32.0,3.0,0.477273
286.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,NaN,Tc,36.0,4.0,0.541667
296.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Authentic,C,CTB,38.0,1.0,0.560606
328.0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,Evaded Clausula Vera,A,tC,42.0,1.0,0.621212


# 7 Corpus Methods

## 7a Build a Corpus

Here we obtain the MEI files from our work.


```python
# now specify YOUR pieces:
my_pieces = ['https://crimproject.org/mei/CRIM_Model_0019.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_1.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_2.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_3.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_4.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_5.mei']


In [22]:

# now specify YOUR pieces:
my_pieces = ['https://crimproject.org/mei/CRIM_Model_0019.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_1.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_2.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_3.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_4.mei', 
             'https://crimproject.org/mei/CRIM_Mass_0019_5.mei']

# file_list = sorted(glob.glob('MEI/LC*'))
corpus = CorpusBase(my_pieces)


## 7b Interval Settings

The settings:  here we specify the various kinds of intervals and length of ngrams we want to find:

```python
combine_unisons_choice = False
combine_rests_choice = False
kind_choice = 'q'
directed_choice = False
compound_choice = False
exclude_rests = True
ngram_length = 4
```

These are in turn used with the various corpus functions we defined at the start of our notebook (see above).  For example:

`corpus_melodic = corpus_mel(corpus, kind_choice, compound_choice, directed_choice)`

or

`corpus_harmonic = corpus_har(corpus, kind_choice, compound_choice, directed_choice)`

or

`corpus_ngrams = corpus_ngrams(corpus, ngram_length, combine_rests_choice)`



In [23]:
# settings:  here we specify the various kinds of intervals and length of ngrams we want to find
combine_unisons_choice = False
combine_rests_choice = False
kind_choice = 'd'
directed_choice = False
compound_choice = False
exclude_rests = True
ngram_length = 4

## 7e.  Corpus:  Points of Imitation and Specific Ngrams 

Here we show ways to filter Presentation Types to show only those with a given melodic idea.


In [24]:
# now specify YOUR pieces:
# set up various 'corpora' of model + mass movements
je_suis = ['https://crimproject.org/mei/CRIM_Model_0009.mei',
       'https://crimproject.org/mei/CRIM_Mass_0006_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0006_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0006_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0006_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0006_5.mei',
          'https://crimproject.org/mei/CRIM_Mass_0007_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0007_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0007_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0007_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0007_5.mei']

vidi_speciosam = ['https://crimproject.org/mei/CRIM_Model_0001.mei',
       'https://crimproject.org/mei/CRIM_Mass_0002_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0002_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0003_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0004_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0005_5.mei']

quo_abiit = ['https://crimproject.org/mei/CRIM_Model_0015.mei',
       'https://crimproject.org/mei/CRIM_Mass_0013_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0013_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0013_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0013_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0013_5.mei']

congratulamini =  ['https://crimproject.org/mei/CRIM_Model_0026.mei',
       'https://crimproject.org/mei/CRIM_Mass_0024_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0024_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0024_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0024_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0024_5.mei']


voulant =  ['https://crimproject.org/mei/CRIM_Model_0012.mei',
       'https://crimproject.org/mei/CRIM_Mass_0010_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0010_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0010_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0010_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0010_5.mei']


veni =  ['https://crimproject.org/mei/CRIM_Model_0019.mei',
       'https://crimproject.org/mei/CRIM_Mass_0019_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0019_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0019_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0019_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0019_5.mei']

ave =  ['https://crimproject.org/mei/CRIM_Model_0008.mei',
       'https://crimproject.org/mei/CRIM_Mass_0005_1.mei',
          'https://crimproject.org/mei/CRIM_Mass_0005_2.mei',
          'https://crimproject.org/mei/CRIM_Mass_0005_3.mei',
          'https://crimproject.org/mei/CRIM_Mass_0005_4.mei',
          'https://crimproject.org/mei/CRIM_Mass_0005_5.mei']

In [25]:
# pick the corpus you want to analyze
my_pieces = veni

In [26]:
# HERE YOU SPECIFY THE CONDITIONS FOR NGRAMS AND PRESENTATION TYPES
melodic_ngram_length=3
end=False
kind='c'
combine_unisons=True 
include_hidden_types=False

# HERE YOU CAN SPECIFY THE INTERVALS YOU WANT TO FIND
my_tuple = ('-3', '3', '2')


for file in my_pieces:
    fileName = file.split("/")[4]
    piece = importScore(file)
    try:
        ptypes = piece.presentationTypes(melodic_ngram_length=melodic_ngram_length, 
                                         end=end, 
                                         kind=kind, 
                                         combine_unisons=combine_unisons, 
                                         include_hidden_types=include_hidden_types)
        if include_hidden_types == False:
            filter_condition = ptypes['Soggetti'].apply(lambda x: my_tuple in x)
            # Apply the filter to get the matching rows
            fitered_ptypes = ptypes[filter_condition]
            cols = ['Composer', 'Title', 'Presentation_Type', 'Soggetti', 'Voices', 
            'Number_Entries', 'Measures_Beats', 'Melodic_Entry_Intervals', 'Time_Entry_Intervals',	
            'Flexed_Entries', 'Parallel_Entries','Parallel_Voice', 'Count_Non_Overlaps', 'Offsets', 'Progress']     
            fitered_ptypes = fitered_ptypes[cols]
            piece.linkExamples(df = fitered_ptypes, piece_url=file)
        if include_hidden_types == True:
            filter_condition = ptypes['Soggetti'].apply(lambda x: my_tuple in x)
            # Apply the filter to get the matching rows
            fitered_ptypes = ptypes[filter_condition]
            cols = ['Composer', 'Title', 'Presentation_Type', 'Soggetti', 'Voices', 
            'Number_Entries', 'Measures_Beats', 'Melodic_Entry_Intervals', 'Time_Entry_Intervals',	
            'Flexed_Entries', 'Parallel_Entries','Parallel_Voice',  'Offsets', 'Progress']     
            fitered_ptypes = fitered_ptypes[cols]
            print(fileName)
            piece.linkExamples(df = fitered_ptypes, piece_url=url)
            
    except:
        print('No P Types found in ' + fileName)
        print('______')
        

,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Veni sponsa Christi,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Model_0019.mei&ema_expression=%2F1%2C11%2C12%2C13%2C14%2C15%2C16%2C17%2C18%2C19%2C2%2C3%2C4%2C6%2C7%2C8%2C9%2F1%2C2%2C2%2C2%2C1%2C1%2C1%2C4%2C4%2C4%2C1%2B2%2C1%2B2%2C2%2C3%2C3%2B4%2C3%2B4%2C4%2F%40all%2C%403.0-end%2C%40all%2C%401-3.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2C%40all%2C%401-1.0%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2C%40all%2C%40all%2B%40all%2C%401-1.0%2B%40all%2C%401-1.0%2Fhighlight&measure_range=1-19&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus, Altus, Cantus, Bassus]",7,"[1.0/1.0, 2.0/1.0, 6.0/1.0, 7.0/1.0, 11.0/3.0, 14.0/1.0, 17.0/1.0]","[P-5, P-4, P-5, P8, P5, P-8]","[8.0, 32.0, 8.0, 36.0, 20.0, 24.0]",False,False,None,0,"[0.0, 8.0, 40.0, 48.0, 84.0, 104.0, 128.0]",0.0


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Kyrie,ID,"[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus]",4,"[1.0/1.0, 2.0/1.0, 6.0/1.0, 7.0/1.0]","[P-5, P-4, P-5]","[8.0, 32.0, 8.0]",False,False,None,0,"[0.0, 8.0, 40.0, 48.0]",0.0


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Gloria,FUGA,"[(-3, 3, 2)]","[Cantus, Tenor]",2,"[1.0/1.0, 4.0/3.0]",[P-8],[28.0],False,False,None,0,"[0.0, 28.0]",0.000000
5,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Gloria,FUGA,"[(-3, 3, 2)]","[Tenor, Cantus, Bassus, Altus, Bassus, Altus]",6,"[44.0/1.0, 45.0/1.0, 46.0/1.0, 47.0/1.0, 54.0/1.0, 55.0/2.0]","[P8, P-12, P8, P-8, P8]","[8.0, 8.0, 8.0, 56.0, 10.0]",False,False,None,1,"[352.0, 360.0, 368.0, 376.0, 432.0, 442.0]",0.444444


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Credo,FUGA,"[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus, Altus]",5,"[1.0/1.0, 2.0/1.0, 6.0/3.0, 7.0/1.0, 10.0/1.0]","[P-5, P-4, P-5, P8]","[8.0, 36.0, 4.0, 24.0]",False,False,None,0,"[0.0, 8.0, 44.0, 48.0, 72.0]",0.0


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Sanctus,FUGA,"[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus]",4,"[1.0/1.0, 3.0/1.0, 7.0/3.0, 12.0/1.0]","[P-5, P-4, P-5]","[16.0, 36.0, 36.0]",False,False,None,0,"[0.0, 16.0, 52.0, 88.0]",0.000000
8,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Sanctus,"<a href=""https://RichardFreedman.github.io/ema_react_app/?pieceURL=https%3A%2F%2Fcrimproject.org%2Fmei%2FCRIM_Mass_0019_4.mei&ema_expression=%2F82%2C83%2C84%2C85%2C86%2C87%2C88%2C89%2C94%2C95%2C96%2C97%2C98%2C99%2F3%2C3%2B4%2C1%2B3%2B4%2C1%2B4%2C1%2C2%2C2%2C2%2C2%2C2%2C2%2C1%2C1%2C1%2F%40all%2C%40all%2B%40all%2C%40all%2B%401-1.0%2B%40all%2C%40all%2B%401-1.0%2C%401-1.0%2C%403.0-end%2C%40all%2C%401-3.0%2C%403.0-end%2C%40all%2C%401-3.0%2C%403.0-end%2C%40all%2C%401-3.0%2Fhighlight&measure_range=82-99&observation=+FUGA"" rel=""noopener noreferrer"" target=""_blank"">FUGA","[(-3, 3, 2)]","[Tenor, Bassus, Cantus, Altus, Altus, Cantus]",6,"[82.0/1.0, 83.0/1.0, 84.0/1.0, 87.0/3.0, 94.0/3.0, 97.0/3.0]","[P-5, P8, P1, P-4, P4]","[8.0, 8.0, 28.0, 56.0, 24.0]",False,False,None,1,"[760.0, 768.0, 776.0, 804.0, 860.0, 884.0]",0.730769


,Composer,Title,Presentation_Type,Soggetti,Voices,Number_Entries,Measures_Beats,Melodic_Entry_Intervals,Time_Entry_Intervals,Flexed_Entries,Parallel_Entries,Parallel_Voice,Count_Non_Overlaps,Offsets,Progress
0,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,FUGA,"[(-3, 3, 2)]","[Cantus, Altus, Tenor, Bassus]",4,"[1.0/1.0, 2.0/1.0, 5.0/3.0, 7.0/3.0]","[P-5, P-4, P-5]","[8.0, 28.0, 16.0]",False,False,None,0,"[0.0, 8.0, 36.0, 52.0]",0.000000
4,Giovanni Pierluigi da Palestrina,Missa Veni sponsa Christi: Agnus Dei,FUGA,"[(-3, 3, 2)]","[Cantus, Altus, Quinta Pars]",3,"[35.0/1.0, 36.0/1.0, 38.0/1.0]","[P-5, P-4]","[8.0, 16.0]",False,False,None,0,"[280.0, 288.0, 304.0]",0.492958


End